# DAICO — Enterprise AI Consultant
### Complete Capstone Notebook

This notebook implements the full **Enterprise AI Consultant** architecture:

**Intake → Router → Selected Specialist Agents → Specialized RAG → Retry → Supervisor → Human Approval → Final Report**

Included:
- Six specialist agents: AI, Software Engineering, Computer Science, IT, Information Systems, Enterprise Risk
- Dynamic routing with `create_agent`, `AgentState`, middleware and tools
- Per-specialty RAG knowledge bases using Hugging Face embeddings + FAISS
- Retry logic with execution logs
- Thread-level memory using `InMemorySaver`
- Human-in-the-loop approval using `interrupt()` / `Command(resume=...)`
- Optional LangSmith tracing
- Professional Gradio UX/UI
- Downloadable Markdown final report

> Run the notebook from top to bottom. The Groq key is requested securely at runtime.
> **This version uses fast dynamic routing in the UI: only Router-selected specialists are invoked, with live step-by-step progress.**


In [1]:
!pip install -q requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [2]:
!pip install -q -U \
    langchain \
    langgraph \
    langchain-groq \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    gradio \
    typing_extensions

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
import re
import time
import uuid
import json
import getpass
import tempfile
from pathlib import Path
from typing import Literal, Callable, Any
from typing_extensions import NotRequired

import gradio as gr

from langchain_groq import ChatGroq
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langchain_core.documents import Document

from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_5626/120303395.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1) API configuration

The Groq key is not stored in the notebook.  
Optional LangSmith tracing can be enabled in the following cell.

In [4]:
import os
from google.colab import userdata

# We use Groq for the agent - add GROQ_API_KEY to Colab Secrets (key icon in left sidebar)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq

# https://console.groq.com/docs/models  (llama-3.3-70b-versatile)
model_llama_70b  = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [6]:
# Optional LangSmith tracing
# Change ENABLE_LANGSMITH to True if you want traces for the capstone demo.

ENABLE_LANGSMITH = False

if ENABLE_LANGSMITH:
    if not os.environ.get("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LANGSMITH_API_KEY: ")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "DEX-Enterprise-AI-Consultant"
    print("LangSmith tracing enabled.")
else:
    print("LangSmith tracing is OFF. Set ENABLE_LANGSMITH=True to enable it.")

LangSmith tracing is OFF. Set ENABLE_LANGSMITH=True to enable it.


## 2) Shared Enterprise State

In [7]:
EnterpriseStep = Literal[
    "intake",
    "router",
    "consultation",
    "supervisor",
    "approval",
    "final",
]

class EnterpriseState(AgentState):
    current_step: NotRequired[EnterpriseStep]

    # Case
    company: NotRequired[str]
    industry: NotRequired[str]
    project: NotRequired[str]
    problem: NotRequired[str]
    requirements: NotRequired[list[str]]
    budget_sar: NotRequired[float]
    expected_users: NotRequired[int]
    additional_context: NotRequired[str]

    # Routing
    selected_agents: NotRequired[list[str]]
    routing_reason: NotRequired[str]

    # Specialist analyses
    ai_analysis: NotRequired[str]
    se_analysis: NotRequired[str]
    cs_analysis: NotRequired[str]
    it_analysis: NotRequired[str]
    is_analysis: NotRequired[str]
    risk_analysis: NotRequired[str]

    # RAG sources
    ai_sources: NotRequired[list[str]]
    se_sources: NotRequired[list[str]]
    cs_sources: NotRequired[list[str]]
    it_sources: NotRequired[list[str]]
    is_sources: NotRequired[list[str]]
    risk_sources: NotRequired[list[str]]

    # Scores
    ai_score: NotRequired[float]
    se_score: NotRequired[float]
    cs_score: NotRequired[float]
    it_score: NotRequired[float]
    is_score: NotRequired[float]
    risk_score: NotRequired[float]

    # Workflow evidence
    execution_log: NotRequired[list[str]]
    retry_log: NotRequired[list[str]]

    # Final
    overall_score: NotRequired[float]
    decision: NotRequired[str]
    final_report: NotRequired[str]
    approval_status: NotRequired[str]
    human_feedback: NotRequired[str]

## 3) Per-specialty RAG

Each specialty has its **own vector store**.  
This means the AI specialist retrieves only AI knowledge, IT retrieves only IT knowledge, etc.

In [8]:
SPECIALTIES = [
    "AI",
    "SOFTWARE_ENGINEERING",
    "COMPUTER_SCIENCE",
    "IT",
    "INFORMATION_SYSTEMS",
    "RISK",
]

vector_stores: dict[str, Any] = {name: None for name in SPECIALTIES}
knowledge_stats: dict[str, dict] = {
    name: {"files": 0, "chunks": 0, "filenames": []}
    for name in SPECIALTIES
}

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
def _normalize_file_list(files):
    if files is None:
        return []
    if isinstance(files, str):
        return [files]
    return list(files)


def load_documents_from_files(files):
    documents = []
    filenames = []

    for file_path in _normalize_file_list(files):
        path = str(file_path)
        suffix = Path(path).suffix.lower()
        filenames.append(Path(path).name)

        if suffix == ".pdf":
            loaded = PyPDFLoader(path).load()
        elif suffix in {".txt", ".md"}:
            loaded = TextLoader(path, autodetect_encoding=True).load()
        else:
            raise ValueError(
                f"Unsupported file type: {suffix}. Use PDF, TXT, or MD."
            )

        for doc in loaded:
            doc.metadata["filename"] = Path(path).name
        documents.extend(loaded)

    return documents, filenames


def index_specialty_files(specialty: str, files):
    specialty = specialty.upper()

    if specialty not in vector_stores:
        raise ValueError(f"Unknown specialty: {specialty}")

    docs, filenames = load_documents_from_files(files)

    if not docs:
        vector_stores[specialty] = None
        knowledge_stats[specialty] = {
            "files": 0,
            "chunks": 0,
            "filenames": [],
        }
        return 0

    chunks = text_splitter.split_documents(docs)
    vector_stores[specialty] = FAISS.from_documents(chunks, embeddings)

    knowledge_stats[specialty] = {
        "files": len(filenames),
        "chunks": len(chunks),
        "filenames": filenames,
    }

    return len(chunks)


def retrieve_specialty_context(
    specialty: str,
    query: str,
    k: int = 4,
    max_attempts: int = 2,
):
    store = vector_stores.get(specialty)

    if store is None:
        return (
            "No specialty knowledge base is currently indexed. "
            "Use professional reasoning and clearly mark assumptions.",
            [],
            0,
        )

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            docs = store.similarity_search(query, k=k)
            context_parts = []
            sources = []

            for i, doc in enumerate(docs, start=1):
                filename = doc.metadata.get(
                    "filename",
                    doc.metadata.get("source", "knowledge_source"),
                )
                page = doc.metadata.get("page")
                source_label = (
                    f"{filename} (page {page + 1})"
                    if isinstance(page, int)
                    else str(filename)
                )
                sources.append(source_label)
                context_parts.append(
                    f"[Source {i}: {source_label}]\n{doc.page_content}"
                )

            return "\n\n".join(context_parts), list(dict.fromkeys(sources)), attempt

        except Exception as exc:
            last_error = exc
            time.sleep(0.5 * attempt)

    raise RuntimeError(
        f"RAG retrieval failed for {specialty} after {max_attempts} attempts: {last_error}"
    )

In [10]:
# Optional starter knowledge so the RAG flow can be demonstrated
# even before you upload your final capstone references.
# Replace/augment this with course PDFs and approved enterprise sources.

STARTER_KNOWLEDGE = {
    "AI": [
        "RAG grounds model answers in retrieved enterprise documents and is useful when knowledge changes frequently. Evaluation should measure retrieval quality, answer faithfulness, task success, safety, and escalation behavior.",
        "Fine-tuning changes model behavior or task adaptation, while RAG supplies external knowledge at inference time. Guardrails and human escalation are important for high-impact enterprise workflows.",
    ],
    "SOFTWARE_ENGINEERING": [
        "Enterprise systems should define functional and non-functional requirements, API contracts, testing layers, observability, maintainability, security boundaries, and deployment practices.",
        "Architecture choice should follow system scale and organizational needs. Modular monoliths can reduce operational complexity; microservices can help independent scaling but add distributed-system overhead.",
    ],
    "COMPUTER_SCIENCE": [
        "Retrieval systems can combine dense vector similarity, metadata filtering, lexical search, and reranking. Evaluation may include precision, recall, MRR, nDCG, latency, and end-to-end answer quality.",
        "Algorithmic design should consider computational complexity, data structures, caching, indexing, latency constraints, and performance bottlenecks.",
    ],
    "IT": [
        "Enterprise infrastructure planning includes compute, storage, networking, monitoring, logging, backups, disaster recovery, high availability, identity controls, and scaling strategy.",
        "Cloud deployment can provide elastic capacity and managed services, while on-premise deployment may be preferred for some sovereignty, integration, or operational constraints.",
    ],
    "INFORMATION_SYSTEMS": [
        "Information systems analysis aligns technology with business processes, stakeholders, roles, data flows, CRM/ERP integrations, governance, organizational change, and measurable business value.",
        "Successful enterprise adoption requires process redesign, ownership, training, access controls, and integration with existing information flows.",
    ],
    "RISK": [
        "Enterprise risk analysis distinguishes operational, financial, model, compliance, security, and adoption risks. Scenario analysis should separate known facts from assumptions.",
        "Expected loss can be estimated as probability of an adverse event multiplied by impact when credible probability and impact estimates are available.",
    ],
}


def seed_starter_knowledge():
    for specialty, texts in STARTER_KNOWLEDGE.items():
        docs = [
            Document(
                page_content=text,
                metadata={"filename": f"starter_{specialty.lower()}.md"},
            )
            for text in texts
        ]
        chunks = text_splitter.split_documents(docs)
        vector_stores[specialty] = FAISS.from_documents(chunks, embeddings)
        knowledge_stats[specialty] = {
            "files": 1,
            "chunks": len(chunks),
            "filenames": [f"starter_{specialty.lower()}.md"],
        }

    return "Starter RAG knowledge indexed for all six specialties."


print(seed_starter_knowledge())

Starter RAG knowledge indexed for all six specialties.


## 4) Specialist Agents

In [11]:
AI_SPECIALIST_PROMPT = '''
You are the AI Specialist in an Enterprise AI Consulting team.

Focus on AI suitability, model strategy, RAG, fine-tuning, hallucinations,
guardrails, evaluation, multilingual requirements, privacy boundaries,
human escalation and AI architecture.

Use the supplied SPECIALTY KNOWLEDGE CONTEXT when it is relevant.
Do not claim a source says something it does not say.
Clearly separate facts from assumptions.

End exactly with:
AI_SCORE: <number from 0 to 100>
'''

SE_SPECIALIST_PROMPT = '''
You are the Software Engineering Specialist.

Focus on requirements, architecture, APIs, frontend/backend boundaries,
testing, maintainability, scalability, integration, SDLC, observability
and engineering tradeoffs.

Use the supplied SPECIALTY KNOWLEDGE CONTEXT when relevant.

End exactly with:
SE_SCORE: <number from 0 to 100>
'''

CS_SPECIALIST_PROMPT = '''
You are the Computer Science Specialist.

Focus only on algorithmic and computational aspects: retrieval,
ranking, search, vector similarity, metadata filtering, reranking,
data structures, optimization, complexity, latency and evaluation.

Avoid generic software-engineering duplication.
Use the supplied SPECIALTY KNOWLEDGE CONTEXT when relevant.

End exactly with:
CS_SCORE: <number from 0 to 100>
'''

IT_SPECIALIST_PROMPT = '''
You are the IT Infrastructure Specialist.

Focus on cloud/on-premise tradeoffs, compute, GPU needs, networking,
storage, databases, deployment, containers, monitoring, logging,
availability, backup, disaster recovery, infrastructure security and scaling.

Use the supplied SPECIALTY KNOWLEDGE CONTEXT when relevant.

End exactly with:
IT_SCORE: <number from 0 to 100>
'''

IS_SPECIALIST_PROMPT = '''
You are the Information Systems Specialist.

Focus on business requirements, business processes, CRM/ERP integration,
information flow, data ownership, users, stakeholders, organizational impact,
change management, governance and business value.

Use the supplied SPECIALTY KNOWLEDGE CONTEXT when relevant.

End exactly with:
IS_SCORE: <number from 0 to 100>
'''

RISK_SPECIALIST_PROMPT = '''
You are the Enterprise Risk and Actuarial Specialist.

Analyze operational, financial, model, security and implementation risk.
Use quantitative reasoning only when the case provides enough information.
Do not invent probabilities. If inputs are missing, use clearly labelled
scenario analysis and state which values are assumptions.

Risk score meaning:
100 = low risk / highly feasible
0 = extreme risk / infeasible

Use the supplied SPECIALTY KNOWLEDGE CONTEXT when relevant.

End exactly with:
RISK_SCORE: <number from 0 to 100>
'''

In [12]:
ai_specialist_agent = create_agent(
    model=model_llama_70b,
    tools=[],
    system_prompt=AI_SPECIALIST_PROMPT,
    name="ai_specialist",
)

software_engineering_agent = create_agent(
    model=model_llama_70b,
    tools=[],
    system_prompt=SE_SPECIALIST_PROMPT,
    name="software_engineering_specialist",
)

computer_science_agent = create_agent(
    model=model_llama_70b,
    tools=[],
    system_prompt=CS_SPECIALIST_PROMPT,
    name="computer_science_specialist",
)

it_specialist_agent = create_agent(
    model=model_llama_70b,
    tools=[],
    system_prompt=IT_SPECIALIST_PROMPT,
    name="it_specialist",
)

information_systems_agent = create_agent(
    model=model_llama_70b,
    tools=[],
    system_prompt=IS_SPECIALIST_PROMPT,
    name="information_systems_specialist",
)

risk_specialist_agent = create_agent(
    model=model_llama_70b,
    tools=[],
    system_prompt=RISK_SPECIALIST_PROMPT,
    name="risk_specialist",
)

SPECIALIST_AGENT_OBJECTS = {
    "AI": ai_specialist_agent,
    "SOFTWARE_ENGINEERING": software_engineering_agent,
    "COMPUTER_SCIENCE": computer_science_agent,
    "IT": it_specialist_agent,
    "INFORMATION_SYSTEMS": information_systems_agent,
    "RISK": risk_specialist_agent,
}

SCORE_KEYWORDS = {
    "AI": "AI_SCORE",
    "SOFTWARE_ENGINEERING": "SE_SCORE",
    "COMPUTER_SCIENCE": "CS_SCORE",
    "IT": "IT_SCORE",
    "INFORMATION_SYSTEMS": "IS_SCORE",
    "RISK": "RISK_SCORE",
}

ANALYSIS_FIELDS = {
    "AI": "ai_analysis",
    "SOFTWARE_ENGINEERING": "se_analysis",
    "COMPUTER_SCIENCE": "cs_analysis",
    "IT": "it_analysis",
    "INFORMATION_SYSTEMS": "is_analysis",
    "RISK": "risk_analysis",
}

SCORE_FIELDS = {
    "AI": "ai_score",
    "SOFTWARE_ENGINEERING": "se_score",
    "COMPUTER_SCIENCE": "cs_score",
    "IT": "it_score",
    "INFORMATION_SYSTEMS": "is_score",
    "RISK": "risk_score",
}

SOURCE_FIELDS = {
    "AI": "ai_sources",
    "SOFTWARE_ENGINEERING": "se_sources",
    "COMPUTER_SCIENCE": "cs_sources",
    "IT": "it_sources",
    "INFORMATION_SYSTEMS": "is_sources",
    "RISK": "risk_sources",
}

## 5) Helpers + Retry

In [13]:
def extract_score(text: str, keyword: str) -> float:
    match = re.search(
        rf"{re.escape(keyword)}\s*:\s*(\d+(?:\.\d+)?)",
        text or "",
        re.IGNORECASE,
    )
    if not match:
        return 50.0
    return max(0.0, min(100.0, float(match.group(1))))


def get_agent_text(result) -> str:
    messages = result.get("messages", [])
    return messages[-1].content if messages else ""


def append_log(state, field: str, entry: str):
    return list(state.get(field, [])) + [entry]


def build_case_from_state(state):
    return f'''
ENTERPRISE CASE

Company: {state.get("company", "Unknown")}
Industry: {state.get("industry", "Unknown")}
Project: {state.get("project", "Unknown")}
Problem: {state.get("problem", "Unknown")}
Requirements: {state.get("requirements", [])}
Budget: {state.get("budget_sar", "Unknown")} SAR
Expected Users: {state.get("expected_users", "Unknown")}
Additional Context: {state.get("additional_context", "")}
'''


def build_retrieval_query(state):
    return " | ".join([
        str(state.get("industry", "")),
        str(state.get("project", "")),
        str(state.get("problem", "")),
        ", ".join(state.get("requirements", []) or []),
    ])


def invoke_with_retry(agent, message: str, max_attempts: int = 3):
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=message)]}
            )
            return result, attempt
        except Exception as exc:
            last_error = exc
            if attempt < max_attempts:
                time.sleep(0.8 * attempt)

    raise RuntimeError(
        f"Specialist failed after {max_attempts} attempts: {last_error}"
    )

## 6) Workflow Tools

In [14]:
@tool
def record_enterprise_case(
    company: str,
    industry: str,
    project: str,
    problem: str,
    requirements: list[str],
    budget_sar: float,
    expected_users: int,
    additional_context: str,
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Store the structured enterprise case and move to routing.'''

    return Command(
        update={
            "company": company,
            "industry": industry,
            "project": project,
            "problem": problem,
            "requirements": requirements,
            "budget_sar": budget_sar,
            "expected_users": expected_users,
            "additional_context": additional_context,
            "current_step": "router",
            "execution_log": append_log(
                runtime.state,
                "execution_log",
                "Intake Agent completed",
            ),
            "messages": [
                ToolMessage(
                    content=(
                        f"Enterprise case recorded: {company} | {project}"
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


VALID_AGENTS = SPECIALTIES.copy()


@tool
def route_to_specialists(
    selected_agents: list[str],
    routing_reason: str,
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Select only the specialist agents that materially contribute.'''

    cleaned = []
    for name in selected_agents:
        name = name.upper().strip()
        if name in VALID_AGENTS and name not in cleaned:
            cleaned.append(name)

    if not cleaned:
        cleaned = ["AI", "SOFTWARE_ENGINEERING"]

    return Command(
        update={
            "selected_agents": cleaned,
            "routing_reason": routing_reason,
            "current_step": "consultation",
            "execution_log": append_log(
                runtime.state,
                "execution_log",
                f"Router selected: {', '.join(cleaned)}",
            ),
            "messages": [
                ToolMessage(
                    content=(
                        f"Selected specialists: {cleaned}\n"
                        f"Routing reason: {routing_reason}"
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [15]:
def run_specialist_tool(
    specialty: str,
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    selected = runtime.state.get("selected_agents", [])

    if specialty not in selected:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content=f"{specialty} was not selected by the Router.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    query = build_retrieval_query(runtime.state)
    context, sources, retrieval_attempts = retrieve_specialty_context(
        specialty,
        query,
        k=4,
        max_attempts=2,
    )

    message = (
        build_case_from_state(runtime.state)
        + "\n\nSPECIALTY KNOWLEDGE CONTEXT:\n"
        + context
        + "\n\n"
        + "Base your analysis on the enterprise case. "
        + "Use retrieved context only where relevant."
    )

    specialist_result, llm_attempts = invoke_with_retry(
        SPECIALIST_AGENT_OBJECTS[specialty],
        message,
        max_attempts=3,
    )

    analysis = get_agent_text(specialist_result)
    score = extract_score(
        analysis,
        SCORE_KEYWORDS[specialty],
    )

    updates = {
        ANALYSIS_FIELDS[specialty]: analysis,
        SCORE_FIELDS[specialty]: score,
        SOURCE_FIELDS[specialty]: sources,
        "execution_log": append_log(
            runtime.state,
            "execution_log",
            f"{specialty} Specialist completed",
        ),
        "retry_log": append_log(
            runtime.state,
            "retry_log",
            (
                f"{specialty}: retrieval attempts={retrieval_attempts}, "
                f"LLM attempts={llm_attempts}"
            ),
        ),
        "messages": [
            ToolMessage(
                content=(
                    f"{specialty} Specialist completed.\n"
                    f"Score: {score}/100\n"
                    f"RAG sources: {sources}\n\n"
                    f"{analysis}"
                ),
                tool_call_id=runtime.tool_call_id,
            )
        ],
    }

    return Command(update=updates)


@tool
def consult_ai_specialist(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Delegate to the AI Specialist.'''
    return run_specialist_tool("AI", runtime)


@tool
def consult_software_engineering(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Delegate to the Software Engineering Specialist.'''
    return run_specialist_tool("SOFTWARE_ENGINEERING", runtime)


@tool
def consult_computer_science(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Delegate to the Computer Science Specialist.'''
    return run_specialist_tool("COMPUTER_SCIENCE", runtime)


@tool
def consult_it_specialist(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Delegate to the IT Specialist.'''
    return run_specialist_tool("IT", runtime)


@tool
def consult_information_systems(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Delegate to the Information Systems Specialist.'''
    return run_specialist_tool("INFORMATION_SYSTEMS", runtime)


@tool
def consult_risk_specialist(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Delegate to the Enterprise Risk / Actuarial Specialist.'''
    return run_specialist_tool("RISK", runtime)

In [16]:
@tool
def finish_specialist_consultation(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''Move to Supervisor only after all selected specialists have completed.'''

    selected = runtime.state.get("selected_agents", [])
    missing = []

    for specialty in selected:
        field = ANALYSIS_FIELDS[specialty]
        if not runtime.state.get(field):
            missing.append(specialty)

    if missing:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content=f"Missing specialist analyses: {missing}",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    return Command(
        update={
            "current_step": "supervisor",
            "execution_log": append_log(
                runtime.state,
                "execution_log",
                "Specialist consultation completed",
            ),
            "messages": [
                ToolMessage(
                    content="All selected specialists completed. Proceed to Supervisor.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

## 7) Deterministic Scoring

In [17]:
SCORE_WEIGHTS = {
    "AI": 0.20,
    "SOFTWARE_ENGINEERING": 0.20,
    "COMPUTER_SCIENCE": 0.10,
    "IT": 0.15,
    "INFORMATION_SYSTEMS": 0.15,
    "RISK": 0.20,
}


def calculate_score(state) -> float:
    selected = state.get("selected_agents", [])
    weighted_total = 0.0
    used_weight = 0.0

    for specialty in selected:
        score = state.get(SCORE_FIELDS[specialty])
        weight = SCORE_WEIGHTS[specialty]

        if score is not None:
            weighted_total += float(score) * weight
            used_weight += weight

    if used_weight == 0:
        return 0.0

    return round(weighted_total / used_weight, 2)


def determine_decision(score: float) -> str:
    if score >= 80:
        return "RECOMMENDED"
    if score >= 65:
        return "RECOMMENDED WITH CONDITIONS"
    if score >= 50:
        return "REQUIRES MAJOR REVISION"
    return "NOT RECOMMENDED"

## 8) Supervisor + Human-in-the-Loop

In [18]:
@tool
def save_draft_report(
    report_body: str,
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''
    Save the Supervisor draft, calculate deterministic score/decision,
    and send the workflow to human approval.
    '''

    score = calculate_score(runtime.state)
    decision = determine_decision(score)

    report = (
        report_body.strip()
        + f"\n\n## 13. Overall Score\n**{score}/100**"
        + f"\n\n## 14. Final Decision\n**{decision}**"
    )

    return Command(
        update={
            "overall_score": score,
            "decision": decision,
            "final_report": report,
            "current_step": "approval",
            "execution_log": append_log(
                runtime.state,
                "execution_log",
                f"Supervisor prepared draft report ({score}/100, {decision})",
            ),
            "messages": [
                ToolMessage(
                    content=(
                        "Draft report saved. "
                        f"Deterministic score={score}/100; decision={decision}. "
                        "Proceed to human approval."
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


@tool
def request_human_approval(
    runtime: ToolRuntime[None, EnterpriseState],
) -> Command:
    '''
    Pause the workflow for enterprise human approval.
    Resume with {"approved": true/false, "feedback": "..."}.
    '''

    response = interrupt({
        "type": "enterprise_approval",
        "message": "Enterprise approval required before finalizing the report.",
        "company": runtime.state.get("company"),
        "project": runtime.state.get("project"),
        "overall_score": runtime.state.get("overall_score"),
        "decision": runtime.state.get("decision"),
        "draft_report": runtime.state.get("final_report"),
    })

    if isinstance(response, dict):
        approved = bool(response.get("approved"))
        feedback = str(response.get("feedback", "")).strip()
    else:
        approved = bool(response)
        feedback = ""

    if approved:
        return Command(
            update={
                "approval_status": "approved",
                "human_feedback": feedback,
                "current_step": "final",
                "execution_log": append_log(
                    runtime.state,
                    "execution_log",
                    "Human reviewer approved the report",
                ),
                "messages": [
                    ToolMessage(
                        content="Human approval received. Finalize the report.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ],
            }
        )

    return Command(
        update={
            "approval_status": "revision_requested",
            "human_feedback": feedback or "Revise the report before approval.",
            "current_step": "supervisor",
            "execution_log": append_log(
                runtime.state,
                "execution_log",
                "Human reviewer requested revision",
            ),
            "messages": [
                ToolMessage(
                    content=(
                        "Human reviewer requested revision. "
                        f"Feedback: {feedback or 'Revise the report.'}"
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

## 9) Stage Prompts + Middleware

In [19]:
INTAKE_PROMPT = '''
You are the Intake Agent for DEX.

Extract the enterprise case and call record_enterprise_case exactly once.

Required fields:
- company
- industry
- project
- problem
- requirements
- budget_sar
- expected_users
- additional_context

Do not analyze the project.
Do not route specialists.
If budget or expected users are unknown, use 0 rather than inventing a value.
'''


ROUTER_PROMPT = '''
You are the Router Agent for DEX.

Case:
Company: {company}
Industry: {industry}
Project: {project}
Problem: {problem}
Requirements: {requirements}
Budget: {budget_sar} SAR
Expected users: {expected_users}

Available specialists:
AI — models, RAG, evaluation, hallucination, guardrails.
SOFTWARE_ENGINEERING — architecture, APIs, testing, maintainability, scalability.
COMPUTER_SCIENCE — algorithms, retrieval, ranking, optimization.
IT — cloud, infrastructure, networking, deployment, monitoring.
INFORMATION_SYSTEMS — business processes, CRM/ERP, enterprise integration.
RISK — operational/financial/model risk, uncertainty, cost-benefit.

Select ONLY specialists that materially contribute.
Do not automatically select all six.

Call route_to_specialists exactly once, then stop.
'''


CONSULTATION_PROMPT = '''
You are the Consultation Orchestrator.

Router-selected specialists:
{selected_agents}

Delegate ONLY to the selected specialists using their matching tools.
Call each selected specialist exactly once.
Do not perform their specialty analysis yourself.
After every selected specialist has completed, call finish_specialist_consultation.
'''


SUPERVISOR_PROMPT = '''
You are the Supervisor Agent.

Enterprise case:
Company: {company}
Industry: {industry}
Project: {project}
Problem: {problem}
Requirements: {requirements}
Budget: {budget_sar} SAR
Expected users: {expected_users}

Selected specialists: {selected_agents}
Routing reason: {routing_reason}

AI Analysis:
{ai_analysis}

Software Engineering Analysis:
{se_analysis}

Computer Science Analysis:
{cs_analysis}

IT Analysis:
{it_analysis}

Information Systems Analysis:
{is_analysis}

Risk Analysis:
{risk_analysis}

Human revision feedback, if any:
{human_feedback}

Synthesize the specialist results. Identify agreements, conflicts,
key conditions, risks and an implementation path.

Do NOT invent the numerical overall score or final decision.
Those are calculated by Python.

Write report_body with ONLY sections 1 through 12:

1. Executive Summary
2. Project Understanding
3. Specialist Findings
4. Technical Feasibility
5. AI Architecture
6. Software Architecture
7. IT Infrastructure
8. Business Integration
9. Risk Assessment
10. Key Conditions
11. Recommended Architecture
12. Implementation Roadmap

Then call save_draft_report(report_body=...).
Do not print a separate final report after the tool call.
'''


APPROVAL_PROMPT = '''
You are at DEX's enterprise approval gate.

Draft report:
{final_report}

Score: {overall_score}/100
Decision: {decision}

You MUST call request_human_approval.
Do not finalize the report before approval.
'''


FINAL_PROMPT = '''
You are DEX's final response stage.

The enterprise report has been approved.

Approved report:
{final_report}

If the latest user message is a new follow-up question, answer it using
the stored enterprise case, specialist analyses and approved report.
Otherwise return the approved report exactly and do not alter its score
or decision.
'''

In [20]:
STEP_CONFIG = {
    "intake": {
        "prompt": INTAKE_PROMPT,
        "tools": [record_enterprise_case],
        "requires": [],
    },
    "router": {
        "prompt": ROUTER_PROMPT,
        "tools": [route_to_specialists],
        "requires": ["company", "project", "problem"],
    },
    "consultation": {
        "prompt": CONSULTATION_PROMPT,
        "tools": [
            consult_ai_specialist,
            consult_software_engineering,
            consult_computer_science,
            consult_it_specialist,
            consult_information_systems,
            consult_risk_specialist,
            finish_specialist_consultation,
        ],
        "requires": ["selected_agents"],
    },
    "supervisor": {
        "prompt": SUPERVISOR_PROMPT,
        "tools": [save_draft_report],
        "requires": ["selected_agents"],
    },
    "approval": {
        "prompt": APPROVAL_PROMPT,
        "tools": [request_human_approval],
        "requires": ["final_report", "overall_score", "decision"],
    },
    "final": {
        "prompt": FINAL_PROMPT,
        "tools": [],
        "requires": ["final_report", "approval_status"],
    },
}


@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    current_step = request.state.get("current_step", "intake")
    stage_config = STEP_CONFIG[current_step]

    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(
                f"{key} must be set before reaching stage '{current_step}'."
            )

    prompt_state = dict(request.state)

    defaults = {
        "company": "",
        "industry": "",
        "project": "",
        "problem": "",
        "requirements": [],
        "budget_sar": "",
        "expected_users": "",
        "selected_agents": [],
        "routing_reason": "",
        "ai_analysis": "Not selected",
        "se_analysis": "Not selected",
        "cs_analysis": "Not selected",
        "it_analysis": "Not selected",
        "is_analysis": "Not selected",
        "risk_analysis": "Not selected",
        "overall_score": "",
        "decision": "",
        "final_report": "",
        "approval_status": "",
        "human_feedback": "",
    }

    for key, value in defaults.items():
        if prompt_state.get(key) is None:
            prompt_state[key] = value

    system_prompt = stage_config["prompt"].format(**prompt_state)

    request = request.override(
        system_prompt=system_prompt,
        tools=stage_config["tools"],
    )

    return handler(request)

## 10) Build the Main Orchestrator

In [21]:
all_tools = [
    record_enterprise_case,
    route_to_specialists,
    consult_ai_specialist,
    consult_software_engineering,
    consult_computer_science,
    consult_it_specialist,
    consult_information_systems,
    consult_risk_specialist,
    finish_specialist_consultation,
    save_draft_report,
    request_human_approval,
]

checkpointer = InMemorySaver()

enterprise_agent = create_agent(
    model=model_llama_70b,
    tools=all_tools,
    state_schema=EnterpriseState,
    middleware=[apply_step_config],
    checkpointer=checkpointer,
    name="DEX_enterprise_ai_consultant",
)

print("DEX agent created successfully.")

DEX agent created successfully.


## 11) Quick Programmatic Demo

The first invocation should pause at the **human approval gate**.  
Use the same `thread_id` to resume.

In [22]:
demo_case = '''
Company: Saudi FinTech Company
Industry: FinTech
Project: AI Customer Support Platform

Problem:
Customer support receives approximately 5,000 requests per day.

Requirements:
- Arabic and English
- 24/7 support
- RAG over internal company policies
- CRM integration
- Human escalation
- Customer data protection
- Scale to approximately 100,000 customers

Budget: 500,000 SAR
Expected users: 100,000

Analyze whether the project is technically, operationally and financially feasible.
'''

demo_thread_id = "DEX-demo-001"
demo_config = {"configurable": {"thread_id": demo_thread_id}}

# Uncomment to run the console demo:
#
# demo_result = enterprise_agent.invoke(
#     {
#         "messages": [HumanMessage(content=demo_case)],
#         "execution_log": [],
#         "retry_log": [],
#     },
#     config=demo_config,
# )
#
# print("Interrupted:", "__interrupt__" in demo_result)
# print("Draft decision:", demo_result.get("decision"))
# print("Draft score:", demo_result.get("overall_score"))
#
# To approve:
# approved = enterprise_agent.invoke(
#     Command(resume={"approved": True, "feedback": ""}),
#     config=demo_config,
# )
# print(approved["messages"][-1].content)

# 12) Professional Gradio UX/UI

The UI contains:
- Structured enterprise case intake
- Knowledge-base upload per specialty
- Automatic agent routing
- Score + decision dashboard
- Specialist analysis tabs
- Human approval / revision workflow
- Trace + retry evidence
- Follow-up questions using the same thread memory
- Downloadable report

In [23]:
# 12) Fast dynamic routing engine + market scoring + evidence + follow-up
# The working dynamic Router/FAST_SESSIONS architecture is preserved.

import traceback
from langchain.messages import SystemMessage

FAST_SESSIONS = {}

AGENT_LABELS = {
    "AI": "AI",
    "SOFTWARE_ENGINEERING": "Software Engineering",
    "COMPUTER_SCIENCE": "Computer Science",
    "IT": "IT Infrastructure",
    "INFORMATION_SYSTEMS": "Information Systems",
    "RISK": "Risk / Actuarial",
}

FEASIBILITY_WEIGHTS = {
    "business_value_roi": 0.20,
    "technical_feasibility": 0.20,
    "data_readiness": 0.15,
    "security_risk_compliance": 0.20,
    "architecture_scalability": 0.10,
    "operational_readiness": 0.10,
    "cost_budget_fit": 0.05,
}

FEASIBILITY_LABELS = {
    "business_value_roi": "Business Value & ROI",
    "technical_feasibility": "Technical Feasibility",
    "data_readiness": "Data Readiness",
    "security_risk_compliance": "Security, Risk & Compliance",
    "architecture_scalability": "Architecture & Scalability",
    "operational_readiness": "Operational Readiness",
    "cost_budget_fit": "Cost & Budget Fit",
}


def _safe_json(text):
    text = (text or "").strip()
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.S)
        if match:
            return json.loads(match.group(0))
        raise


def build_ui_case(company, industry, project, problem, requirements, budget, expected_users, additional_context):
    req_lines = [
        line.strip("-• ").strip()
        for line in (requirements or "").splitlines()
        if line.strip()
    ]
    return {
        "company": (company or "Unknown").strip(),
        "industry": (industry or "Unknown").strip(),
        "project": (project or "Unknown").strip(),
        "problem": (problem or "Not provided").strip(),
        "requirements": req_lines,
        "budget_sar": float(budget or 0),
        "expected_users": int(expected_users or 0),
        "additional_context": (additional_context or "").strip(),
        "execution_log": [],
        "retry_log": [],
    }


def build_case_text_from_dict(state):
    req = "\n".join(f"- {x}" for x in state.get("requirements", [])) or "- Not provided"
    return f"""Company: {state.get('company', 'Unknown')}
Industry: {state.get('industry', 'Unknown')}
Project: {state.get('project', 'Unknown')}
Problem: {state.get('problem', 'Not provided')}
Requirements:
{req}
Budget: {state.get('budget_sar', 0):,.2f} SAR
Expected Users: {state.get('expected_users', 0)}
Additional Context: {state.get('additional_context', '') or 'None'}"""


def fallback_router(state):
    text = build_case_text_from_dict(state).lower()
    selected = []

    if any(k in text for k in ["ai", "rag", "llm", "model", "hallucination", "chatbot"]):
        selected.append("AI")
    if any(k in text for k in ["platform", "software", "api", "backend", "frontend", "testing", "architecture", "application"]):
        selected.append("SOFTWARE_ENGINEERING")
    if any(k in text for k in ["retrieval", "ranking", "vector", "algorithm", "optimization", "search"]):
        selected.append("COMPUTER_SCIENCE")
    if any(k in text for k in ["cloud", "infrastructure", "deployment", "monitoring", "availability", "network", "storage", "scale", "100,000"]):
        selected.append("IT")
    if any(k in text for k in ["crm", "erp", "business process", "integration", "stakeholder", "organizational"]):
        selected.append("INFORMATION_SYSTEMS")
    if any(k in text for k in ["budget", "risk", "financial", "sensitive", "compliance", "customer data", "fintech"]):
        selected.append("RISK")

    if not selected:
        selected = ["SOFTWARE_ENGINEERING", "RISK"]

    return selected[:4], "Fallback keyword routing was used because the structured Router output could not be parsed."


def route_case_dynamic(state):
    router_system = """You are DEX's enterprise routing agent.

Select only the specialists that materially contribute to this case.
Choose between 1 and 4 specialists. Do not select everyone by default.

Available values:
AI
SOFTWARE_ENGINEERING
COMPUTER_SCIENCE
IT
INFORMATION_SYSTEMS
RISK

Return ONLY JSON exactly like:
{"selected_agents": ["AI", "IT"], "routing_reason": "short explanation"}

Selection guidance:
AI = models, RAG, LLM evaluation, hallucinations, guardrails.
SOFTWARE_ENGINEERING = software architecture, APIs, testing, maintainability.
COMPUTER_SCIENCE = algorithms, retrieval/ranking, optimization.
IT = cloud, infrastructure, deployment, monitoring, availability.
INFORMATION_SYSTEMS = CRM/ERP, business process and organizational integration.
RISK = financial/operational/model risk, uncertainty and cost-benefit.
"""
    try:
        response = model_llama_70b.invoke([
            SystemMessage(content=router_system),
            HumanMessage(content=build_case_text_from_dict(state)),
        ])
        data = _safe_json(response.content)

        selected = [
            x for x in data.get("selected_agents", [])
            if x in SPECIALIST_AGENT_OBJECTS
        ]
        selected = list(dict.fromkeys(selected))[:4]

        if not selected:
            raise ValueError("Router returned no valid specialists")

        return selected, data.get(
            "routing_reason",
            "Selected based on enterprise case requirements."
        )

    except Exception:
        return fallback_router(state)


def run_one_specialist_fast(specialty, state):
    query = build_retrieval_query(state)

    context, sources, retrieval_attempts = retrieve_specialty_context(
        specialty,
        query,
        k=3,
        max_attempts=2,
    )

    # Keep the exact retrieval query and text that were passed to the specialist.
    # This makes the RAG pipeline visible in the UI instead of showing source names only.
    state.setdefault("rag_retrieval_details", {})
    state["rag_retrieval_details"][specialty] = {
        "query": query,
        "sources": sources,
        "context": context,
        "retrieval_attempts": retrieval_attempts,
    }

    message = (
        build_case_text_from_dict(state)
        + "\n\nSPECIALTY KNOWLEDGE CONTEXT:\n"
        + context
        + "\n\nAnalyze only from your specialty. "
          "Use retrieved evidence where relevant and clearly identify assumptions."
    )

    specialist_result, llm_attempts = invoke_with_retry(
        SPECIALIST_AGENT_OBJECTS[specialty],
        message,
        max_attempts=2,
    )

    analysis = get_agent_text(specialist_result)
    score = extract_score(analysis, SCORE_KEYWORDS[specialty])

    state[ANALYSIS_FIELDS[specialty]] = analysis
    state[SCORE_FIELDS[specialty]] = score
    state[SOURCE_FIELDS[specialty]] = sources

    state["execution_log"].append(
        f"{specialty} specialist completed"
    )

    state["retry_log"].append(
        f"{specialty}: retrieval attempts={retrieval_attempts}, "
        f"LLM attempts={llm_attempts}"
    )


def _specialist_material(state):
    blocks = []

    for specialty in state.get("selected_agents", []):
        blocks.append(
            f"""
SPECIALIST: {AGENT_LABELS[specialty]}
SPECIALIST SCORE: {state.get(SCORE_FIELDS[specialty], 'N/A')}/100
ANALYSIS:
{state.get(ANALYSIS_FIELDS[specialty], 'No analysis available.')}

RAG SOURCES:
{state.get(SOURCE_FIELDS[specialty], [])}
""".strip()
        )

    return "\n\n".join(blocks)


def _fallback_market_evaluation(state):
    def _avg(values, default=60.0):
        vals = [float(v) for v in values if v is not None]
        return round(sum(vals) / len(vals), 1) if vals else default

    ai = state.get("ai_score")
    se = state.get("se_score")
    cs = state.get("cs_score")
    it = state.get("it_score")
    info = state.get("is_score")
    risk = state.get("risk_score")

    scores = {
        "business_value_roi": _avg([info, risk], 60),
        "technical_feasibility": _avg([ai, se, cs], 60),
        "data_readiness": _avg([ai, info], 55),
        "security_risk_compliance": _avg([risk, it], 55),
        "architecture_scalability": _avg([se, it], 60),
        "operational_readiness": _avg([it, info, risk], 55),
        "cost_budget_fit": _avg([risk, info], 55),
    }

    rationales = {
        key: "Fallback score derived from available specialist scores because the market evaluator output could not be parsed."
        for key in scores
    }

    return {
        "criterion_scores": scores,
        "criterion_rationales": rationales,
        "critical_risks": [],
    }


def evaluate_market_feasibility(state):
    evaluator_system = """You are DEX's independent enterprise feasibility evaluator.

Evaluate the proposed project against a fixed market-oriented rubric.
Use ONLY the enterprise case, specialist findings, and RAG evidence supplied.

Score every criterion from 0 to 100:

1. business_value_roi
   Business value, measurable impact, ROI path, and strategic fit.

2. technical_feasibility
   Whether the required software and AI capabilities are realistically deliverable.

3. data_readiness
   Availability, quality, permissions, governance, freshness, and suitability of required data/documents.

4. security_risk_compliance
   Privacy, cybersecurity, model risk, compliance exposure, human escalation, and governance.

5. architecture_scalability
   Integration, reliability, availability, performance, and ability to scale to the expected workload.

6. operational_readiness
   Monitoring, ownership, maintenance, support, incident response, deployment, backup, and recovery.

7. cost_budget_fit
   Whether the stated budget plausibly fits implementation, integration, infrastructure, model usage, operations, and controls.

Scoring anchors:
90-100 = strong evidence and close to production-ready
75-89  = feasible with manageable gaps
60-74  = material gaps or uncertainty
40-59  = major deficiencies
0-39   = critical blockers or insufficient feasibility

Rules:
- Missing information must NOT receive a high score.
- Be conservative when evidence is incomplete.
- Do not invent costs, laws, probabilities, or company capabilities.
- A critical risk is something that could block safe or viable deployment if left unresolved.
- Mark critical risks as mitigated true only if the case explicitly contains an adequate control.

Return ONLY JSON in this exact structure:
{
  "criterion_scores": {
    "business_value_roi": 0,
    "technical_feasibility": 0,
    "data_readiness": 0,
    "security_risk_compliance": 0,
    "architecture_scalability": 0,
    "operational_readiness": 0,
    "cost_budget_fit": 0
  },
  "criterion_rationales": {
    "business_value_roi": "short rationale",
    "technical_feasibility": "short rationale",
    "data_readiness": "short rationale",
    "security_risk_compliance": "short rationale",
    "architecture_scalability": "short rationale",
    "operational_readiness": "short rationale",
    "cost_budget_fit": "short rationale"
  },
  "critical_risks": [
    {
      "risk": "risk description",
      "severity": "critical",
      "mitigated": false,
      "reason": "why this can block deployment"
    }
  ]
}
"""

    prompt = f"""ENTERPRISE CASE:
{build_case_text_from_dict(state)}

SELECTED SPECIALIST FINDINGS:
{_specialist_material(state)}
"""

    try:
        response = model_llama_70b.invoke([
            SystemMessage(content=evaluator_system),
            HumanMessage(content=prompt),
        ])

        data = _safe_json(response.content)

        raw_scores = data.get("criterion_scores", {})
        scores = {}

        for key in FEASIBILITY_WEIGHTS:
            value = float(raw_scores.get(key, 50))
            scores[key] = max(0.0, min(100.0, value))

        rationales = data.get("criterion_rationales", {})
        critical_risks = data.get("critical_risks", [])

        return {
            "criterion_scores": scores,
            "criterion_rationales": rationales,
            "critical_risks": critical_risks if isinstance(critical_risks, list) else [],
        }

    except Exception:
        return _fallback_market_evaluation(state)


def calculate_market_score(evaluation):
    scores = evaluation.get("criterion_scores", {})

    total = 0.0
    for key, weight in FEASIBILITY_WEIGHTS.items():
        total += float(scores.get(key, 0)) * weight

    return round(total, 2)


def determine_market_decision(score, critical_risks):
    if score >= 80:
        decision = "RECOMMENDED"
    elif score >= 65:
        decision = "RECOMMENDED WITH CONDITIONS"
    else:
        decision = "NOT RECOMMENDED"

    unmitigated_critical = [
        risk for risk in (critical_risks or [])
        if str(risk.get("severity", "")).lower() == "critical"
        and not bool(risk.get("mitigated", False))
    ]

    # Critical Risk Gate:
    # an unresolved critical risk prevents an unconditional RECOMMENDED decision.
    if unmitigated_critical and decision == "RECOMMENDED":
        decision = "RECOMMENDED WITH CONDITIONS"

    return decision


def supervisor_fast(state, feedback=""):
    evaluation = evaluate_market_feasibility(state)
    score = calculate_market_score(evaluation)
    decision = determine_market_decision(
        score,
        evaluation.get("critical_risks", []),
    )

    state["feasibility_scores"] = evaluation["criterion_scores"]
    state["feasibility_rationales"] = evaluation["criterion_rationales"]
    state["critical_risks"] = evaluation["critical_risks"]
    state["overall_score"] = score
    state["decision"] = decision

    system = """You are DEX's Supervisor Agent.

Synthesize the internal specialist work into ONE professional enterprise feasibility report.
Do not expose a separate specialist-by-specialist report to the end user.
Do not invent or change the supplied feasibility scores or final decision.
Do not repeat raw specialist analyses word-for-word.

Use these exact sections:

## 1. Executive Summary
## 2. Technical Assessment
Cover technical feasibility, data readiness, architecture, scalability, infrastructure, AI approach, security, and technical risks.

## 3. Business Assessment
Cover business value, ROI path, business fit, enterprise integration, operational readiness, cost/budget fit, and organizational impact.

## 4. Risk & Compliance
Explain the most important deployment risks and the critical-risk gate.

## 5. Recommended Architecture
## 6. Implementation Roadmap
## 7. Feasibility Score Rationale
Explain the seven rubric criteria without changing their supplied numbers.

## 8. Final Decision
State the supplied decision and the conditions required to proceed.
"""

    prompt = f"""ENTERPRISE CASE:
{build_case_text_from_dict(state)}

INTERNAL ROUTING:
{state.get('selected_agents', [])}

ROUTING REASON:
{state.get('routing_reason', '')}

SPECIALIST MATERIAL:
{_specialist_material(state)}

MARKET FEASIBILITY RUBRIC SCORES:
{json.dumps(state.get('feasibility_scores', {}), indent=2)}

RUBRIC RATIONALES:
{json.dumps(state.get('feasibility_rationales', {}), indent=2)}

CRITICAL RISKS:
{json.dumps(state.get('critical_risks', []), indent=2)}

PROJECT FEASIBILITY SCORE:
{score}/100

FINAL DECISION:
{decision}

REVISION FEEDBACK:
{feedback or 'None'}
"""

    response = model_llama_70b.invoke([
        SystemMessage(content=system),
        HumanMessage(content=prompt),
    ])

    state["final_report"] = response.content
    state["approval_status"] = "Awaiting human review"
    state["execution_log"].append(
        f"Supervisor synthesized final draft using market feasibility rubric ({score}/100, {decision})"
    )

    return state


def dashboard_markdown(state):
    score = state.get("overall_score", "—")
    decision = state.get("decision", "Ready")
    approval = state.get("approval_status", "Not started")
    scores = state.get("feasibility_scores", {})

    rows = []
    for key in FEASIBILITY_WEIGHTS:
        label = FEASIBILITY_LABELS[key]
        value = scores.get(key, "—")
        weight = int(FEASIBILITY_WEIGHTS[key] * 100)
        rows.append(f"| {label} | {value} | {weight}% |")

    table = "\n".join(rows) if rows else "| No evaluation yet | — | — |"

    critical = [
        r for r in state.get("critical_risks", [])
        if str(r.get("severity", "")).lower() == "critical"
        and not bool(r.get("mitigated", False))
    ]

    gate = (
        f"{len(critical)} unresolved critical risk(s) currently limit an unconditional recommendation."
        if critical
        else "No unresolved critical risk gate is currently blocking the decision."
    )

    return f"""## Project Feasibility

### **{score}/100** — {decision}

**Human Review:** {approval}

| Evaluation Criterion | Score | Weight |
|---|---:|---:|
{table}

**Critical Risk Gate:** {gate}
"""


def routing_summary_markdown(state):
    selected = state.get("selected_agents", [])

    if not selected:
        return """### Internal Routing

No specialists have been selected yet.
"""

    names = ", ".join(AGENT_LABELS[a] for a in selected)

    return f"""### Internal Routing

**Selected internal specialists:** {names}

**Routing rationale:** {state.get('routing_reason', '—')}

The specialist agents remain internal. Their findings are consolidated into the Technical and Business assessments shown in the results tab.
"""


def _extract_report_section(report, title, fallback):
    if not report:
        return fallback

    pattern = rf"(?ims)^##\s*\d+\.\s*{re.escape(title)}\s*$\s*(.*?)(?=^##\s*\d+\.\s*|\Z)"
    match = re.search(pattern, report)

    if match:
        return match.group(1).strip()

    return fallback


def technical_assessment_markdown(state):
    report = state.get("final_report", "")

    fallback_parts = []
    for specialty in ["AI", "SOFTWARE_ENGINEERING", "COMPUTER_SCIENCE", "IT"]:
        analysis = state.get(ANALYSIS_FIELDS[specialty])
        if analysis:
            fallback_parts.append(analysis)

    fallback = (
        "\n\n".join(fallback_parts)
        if fallback_parts
        else "Technical assessment will appear after the consultation."
    )

    body = _extract_report_section(
        report,
        "Technical Assessment",
        fallback,
    )

    return f"""## Technical Assessment

{body}
"""


def business_assessment_markdown(state):
    report = state.get("final_report", "")

    fallback_parts = []
    for specialty in ["INFORMATION_SYSTEMS", "RISK"]:
        analysis = state.get(ANALYSIS_FIELDS[specialty])
        if analysis:
            fallback_parts.append(analysis)

    fallback = (
        "\n\n".join(fallback_parts)
        if fallback_parts
        else "Business assessment will appear after the consultation."
    )

    body = _extract_report_section(
        report,
        "Business Assessment",
        fallback,
    )

    return f"""## Business Assessment

{body}
"""


def _clean_rag_preview(text, max_chars=1400):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if not text:
        return "No retrieved text was returned."
    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."
    return text


def rag_evidence_markdown(state):
    selected = state.get("selected_agents", [])
    details = state.get("rag_retrieval_details", {})

    if not selected:
        return """## Retrieval Evidence

Run a consultation to see the exact retrieval query, source files, and retrieved knowledge passed to each selected internal specialist.
"""

    lines = [
        "## Retrieval Evidence",
        "",
        "This section shows what the RAG layer actually retrieved and passed into the specialist analysis.",
    ]

    any_evidence = False

    for specialty in selected:
        label = AGENT_LABELS[specialty]
        item = details.get(specialty, {})
        sources = item.get("sources") or state.get(SOURCE_FIELDS[specialty], []) or []
        context = item.get("context", "")
        query = item.get("query", "")

        lines.append(f"\n### {label}")

        if query:
            lines.append(f"**Retrieval query:** `{query}`")

        if sources:
            any_evidence = True
            lines.append("**Source files retrieved:**")
            for source in sources:
                lines.append(f"- `{source}`")
        else:
            lines.append("**Source files retrieved:** None")

        if context:
            any_evidence = True
            lines.append("\n**Retrieved knowledge passed to the specialist:**")
            lines.append(f"> {_clean_rag_preview(context)}")
        else:
            lines.append(
                "\n**Retrieved knowledge passed to the specialist:** "
                "No custom retrieval text was returned."
            )

    if not any_evidence:
        lines.append(
            "\nNo indexed evidence was retrieved for this consultation. "
            "The analysis therefore relied on the existing fallback/starter knowledge."
        )

    return "\n".join(lines)

def trace_json_fast(state):
    return {
        "selected_agents": state.get("selected_agents", []),
        "routing_reason": state.get("routing_reason", ""),
        "project_feasibility_score": state.get("overall_score"),
        "decision": state.get("decision"),
        "feasibility_scores": state.get("feasibility_scores", {}),
        "critical_risks": state.get("critical_risks", []),
        "execution_log": state.get("execution_log", []),
        "retry_log": state.get("retry_log", []),
        "rag_sources": {
            a: state.get(SOURCE_FIELDS[a], [])
            for a in state.get("selected_agents", [])
        },
        "rag_retrieval_details": state.get("rag_retrieval_details", {}),
        "last_followup_routing": state.get("last_followup_routing", {}),
    }


def result_tuple(state, thread_id, status):
    return (
        status,
        dashboard_markdown(state),
        routing_summary_markdown(state),
        state.get(
            "final_report",
            "The Supervisor report will appear here when the selected analysis is complete."
        ),
        technical_assessment_markdown(state),
        business_assessment_markdown(state),
        rag_evidence_markdown(state),
        trace_json_fast(state),
        f"### Human Review\n**{state.get('approval_status', 'Not started')}**",
        thread_id,
    )


def run_consultation_ui(
    company,
    industry,
    project,
    problem,
    requirements,
    budget,
    expected_users,
    additional_context,
):
    if not project or not problem:
        empty = build_ui_case(
            company,
            industry,
            project,
            problem,
            requirements,
            budget,
            expected_users,
            additional_context,
        )
        yield result_tuple(
            empty,
            "",
            "**Error:** Please fill in Project and Problem first.",
        )
        return

    thread_id = f"DEX-{uuid.uuid4().hex[:10]}"

    state = build_ui_case(
        company,
        industry,
        project,
        problem,
        requirements,
        budget,
        expected_users,
        additional_context,
    )

    FAST_SESSIONS[thread_id] = state

    yield result_tuple(
        state,
        thread_id,
        "**Step 1/4 — Reading the enterprise case...**",
    )

    selected, reason = route_case_dynamic(state)

    state["selected_agents"] = selected
    state["routing_reason"] = reason
    state["execution_log"].append(
        f"Router selected: {', '.join(selected)}"
    )

    FAST_SESSIONS[thread_id] = state

    yield result_tuple(
        state,
        thread_id,
        "**Step 2/4 — Routing complete. Only the required internal specialists will be called.**",
    )

    total = len(selected)

    for index, specialty in enumerate(selected, start=1):
        yield result_tuple(
            state,
            thread_id,
            f"**Step 3/4 — {AGENT_LABELS[specialty]} analysis in progress ({index}/{total})...**",
        )

        try:
            run_one_specialist_fast(specialty, state)
        except Exception as exc:
            state["execution_log"].append(
                f"{specialty} failed: {exc}"
            )
            state[ANALYSIS_FIELDS[specialty]] = (
                f"Analysis failed after retry: {exc}"
            )
            state[SCORE_FIELDS[specialty]] = 50.0

        FAST_SESSIONS[thread_id] = state

        yield result_tuple(
            state,
            thread_id,
            f"**{AGENT_LABELS[specialty]} analysis complete.**",
        )

    yield result_tuple(
        state,
        thread_id,
        "**Step 4/4 — Calculating market feasibility and synthesizing the final report...**",
    )

    state = supervisor_fast(state)
    FAST_SESSIONS[thread_id] = state

    yield result_tuple(
        state,
        thread_id,
        "**Consultation complete. Review the assessment and approve or request a revision.**",
    )


def approve_ui(thread_id):
    state = FAST_SESSIONS.get(thread_id)

    if not state:
        empty = {"execution_log": [], "retry_log": []}
        return result_tuple(
            empty,
            "",
            "**Error:** Run a consultation first.",
        )

    state["approval_status"] = "Approved"
    state["execution_log"].append(
        "Human approved final report"
    )

    FAST_SESSIONS[thread_id] = state

    return result_tuple(
        state,
        thread_id,
        "**Report approved.**",
    )


def revise_ui(thread_id, feedback):
    state = FAST_SESSIONS.get(thread_id)

    if not state:
        empty = {"execution_log": [], "retry_log": []}
        return result_tuple(
            empty,
            "",
            "**Error:** Run a consultation first.",
        )

    if not (feedback or "").strip():
        return result_tuple(
            state,
            thread_id,
            "Please write revision feedback first.",
        )

    state["approval_status"] = "Revising"

    state = supervisor_fast(
        state,
        feedback=feedback.strip(),
    )

    state["execution_log"].append(
        "Human requested report revision"
    )

    FAST_SESSIONS[thread_id] = state

    return result_tuple(
        state,
        thread_id,
        "**Revision complete. Review the updated report.**",
    )


def route_followup_question(state, question):
    router_system = """You are DEX's follow-up routing agent.

Route the user's NEW follow-up question to only the internal specialists
that can add NEW analysis beyond the existing report.

Choose 1 or 2 values only from:
AI
SOFTWARE_ENGINEERING
COMPUTER_SCIENCE
IT
INFORMATION_SYSTEMS
RISK

Examples:
- infrastructure, cloud, monitoring -> IT
- API, architecture, implementation -> SOFTWARE_ENGINEERING
- RAG, model, hallucination, AI quality -> AI
- retrieval, ranking, vector search -> COMPUTER_SCIENCE
- CRM, process, business integration -> INFORMATION_SYSTEMS
- budget, ROI uncertainty, risk, financial impact -> RISK

Return ONLY JSON:
{
  "selected_agents": ["IT"],
  "routing_reason": "short explanation"
}
"""

    prompt = f"""CURRENT ENTERPRISE CASE:
{build_case_text_from_dict(state)}

CURRENT DECISION:
{state.get('decision')}

FOLLOW-UP QUESTION:
{question}
"""

    try:
        response = model_llama_70b.invoke([
            SystemMessage(content=router_system),
            HumanMessage(content=prompt),
        ])

        data = _safe_json(response.content)

        selected = [
            x for x in data.get("selected_agents", [])
            if x in SPECIALIST_AGENT_OBJECTS
        ]

        selected = list(dict.fromkeys(selected))[:2]

        if not selected:
            raise ValueError("No valid follow-up specialists")

        return selected, data.get(
            "routing_reason",
            "Selected based on the follow-up question.",
        )

    except Exception:
        q = question.lower()

        if any(k in q for k in ["cloud", "infrastructure", "monitor", "deploy", "network", "storage"]):
            return ["IT"], "Fallback follow-up routing selected IT."
        if any(k in q for k in ["budget", "cost", "roi", "risk", "financial"]):
            return ["RISK"], "Fallback follow-up routing selected Risk / Actuarial."
        if any(k in q for k in ["crm", "business", "process", "integration"]):
            return ["INFORMATION_SYSTEMS"], "Fallback follow-up routing selected Information Systems."
        if any(k in q for k in ["rag", "llm", "model", "hallucination", "ai"]):
            return ["AI"], "Fallback follow-up routing selected AI."
        if any(k in q for k in ["retrieval", "ranking", "vector", "algorithm"]):
            return ["COMPUTER_SCIENCE"], "Fallback follow-up routing selected Computer Science."

        return ["SOFTWARE_ENGINEERING"], "Fallback follow-up routing selected Software Engineering."


def follow_up_ui(thread_id, question):
    state = FAST_SESSIONS.get(thread_id)

    if not state:
        return (
            "**Error:** Run a consultation first.",
            trace_json_fast({}),
        )

    question = (question or "").strip()

    if not question:
        return (
            "Type a follow-up question first.",
            trace_json_fast(state),
        )

    selected, reason = route_followup_question(
        state,
        question,
    )

    specialist_answers = []
    evidence = {}

    for specialty in selected:
        context, sources, retrieval_attempts = retrieve_specialty_context(
            specialty,
            question,
            k=3,
            max_attempts=2,
        )

        evidence[specialty] = {
            "sources": sources,
            "retrieved_context": _clean_rag_preview(context, max_chars=1800),
        }

        specialist_prompt = f"""CURRENT ENTERPRISE CASE:
{build_case_text_from_dict(state)}

CURRENT PROJECT FEASIBILITY SCORE:
{state.get('overall_score')}/100

CURRENT DECISION:
{state.get('decision')}

USER FOLLOW-UP QUESTION:
{question}

RETRIEVED SPECIALTY EVIDENCE:
{context}

Answer only the NEW follow-up question from your specialty.
Do not repeat the existing report unless needed to explain a change.
If this is a what-if scenario, explain which feasibility criteria would be affected.
Clearly separate evidence from assumptions.
"""

        result, attempts = invoke_with_retry(
            SPECIALIST_AGENT_OBJECTS[specialty],
            specialist_prompt,
            max_attempts=2,
        )

        specialist_answers.append(
            f"### {AGENT_LABELS[specialty]}\n{get_agent_text(result)}"
        )

        state["retry_log"].append(
            f"Follow-up {specialty}: retrieval attempts={retrieval_attempts}, "
            f"LLM attempts={attempts}"
        )

    synthesis_system = """You are DEX's follow-up consultation supervisor.

Answer the user's follow-up as a continuation of the existing enterprise consultation.

Rules:
- Do NOT simply restate the existing report.
- Focus on the NEW question.
- Use the newly routed specialist analysis and newly retrieved RAG evidence.
- If the question is a what-if scenario, explain what changes and what remains unchanged.
- Do not silently replace the stored official score; if the scenario could change it,
  state which rubric criteria should be re-evaluated.
- Be concise, practical, and decision-oriented.
- End with a short "Evidence used" section listing the retrieved source names when available.
"""

    synthesis_prompt = f"""USER QUESTION:
{question}

ROUTING:
{selected}

ROUTING REASON:
{reason}

SPECIALIST FOLLOW-UP ANALYSIS:
{chr(10).join(specialist_answers)}

RAG EVIDENCE:
{json.dumps(evidence, indent=2)}
"""

    response = model_llama_70b.invoke([
        SystemMessage(content=synthesis_system),
        HumanMessage(content=synthesis_prompt),
    ])

    state["last_followup_routing"] = {
        "question": question,
        "selected_agents": selected,
        "routing_reason": reason,
        "rag_sources": evidence,
    }

    state["execution_log"].append(
        f"Follow-up routed to: {', '.join(selected)}"
    )

    FAST_SESSIONS[thread_id] = state

    return response.content, trace_json_fast(state)


def index_all_ui(
    ai_files,
    se_files,
    cs_files,
    it_files,
    is_files,
    risk_files,
):
    try:
        mapping = {
            "AI": ai_files,
            "SOFTWARE_ENGINEERING": se_files,
            "COMPUTER_SCIENCE": cs_files,
            "IT": it_files,
            "INFORMATION_SYSTEMS": is_files,
            "RISK": risk_files,
        }

        rows = []

        for specialty, specialty_files in mapping.items():
            if specialty_files:
                chunks = index_specialty_files(
                    specialty,
                    specialty_files,
                )

                stats = knowledge_stats.get(
                    specialty,
                    {},
                )

                filenames = stats.get(
                    "filenames",
                    [],
                )

                rows.append(
                    f"""### {AGENT_LABELS[specialty]}
- Files indexed: {stats.get('files', len(filenames))}
- Chunks created: {chunks}
- Files: {', '.join(filenames) if filenames else 'None'}
"""
                )

        if not rows:
            return (
                "No new files were selected. "
                "The existing indexed knowledge remains available."
            )

        return (
            "## Knowledge Base Updated\n\n"
            + "\n".join(rows)
            + "\nThese documents will be searched automatically only when the Router calls the relevant internal specialist."
        )

    except Exception as exc:
        return f"**Knowledge indexing error:** {exc}"


In [24]:
# 13) Professional, full-width, Colab-safe Gradio UX/UI

PRO_CSS = """
:root,
html,
body,
.gradio-container {
    color-scheme: dark !important;

    --body-background-fill: #07111f !important;
    --body-background-fill-dark: #07111f !important;

    --background-fill-primary: #07111f !important;
    --background-fill-secondary: #0d1b2c !important;

    --block-background-fill: #0d1b2c !important;
    --block-background-fill-dark: #0d1b2c !important;

    --block-border-color: rgba(255,255,255,.10) !important;
    --block-border-color-dark: rgba(255,255,255,.10) !important;

    --body-text-color: #eef4fb !important;
    --body-text-color-dark: #eef4fb !important;

    --block-label-text-color: #c7d5e6 !important;
    --block-label-text-color-dark: #c7d5e6 !important;

    --input-background-fill: #10233a !important;
    --input-background-fill-dark: #10233a !important;

    --input-border-color: rgba(255,255,255,.12) !important;
    --input-border-color-dark: rgba(255,255,255,.12) !important;

    --button-primary-background-fill: #4f7fc7 !important;
    --button-primary-background-fill-hover: #5f8fd6 !important;
    --button-primary-text-color: #ffffff !important;

    --button-secondary-background-fill: #11243a !important;
    --button-secondary-text-color: #eef4fb !important;
}

html,
body {
    margin: 0 !important;
    padding: 0 !important;
    width: 100% !important;
    min-height: 100% !important;
    background: #07111f !important;
}

body > gradio-app,
gradio-app,
#root {
    width: 100% !important;
    max-width: none !important;
    margin: 0 !important;
    padding: 0 !important;
    background: #07111f !important;
}

.gradio-container {
    width: 100% !important;
    max-width: none !important;
    min-height: 100vh !important;
    margin: 0 !important;
    padding: 24px 32px 40px !important;

    background:
        radial-gradient(circle at 88% 5%, rgba(79,127,199,.10), transparent 25%),
        radial-gradient(circle at 5% 90%, rgba(105,145,190,.07), transparent 27%),
        linear-gradient(145deg, #07111f 0%, #091523 60%, #0a1625 100%) !important;

    color: #eef4fb !important;

    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif !important;
}

.gradio-container,
.gradio-container p,
.gradio-container span,
.gradio-container label,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container li,
.gradio-container td,
.gradio-container th {
    color: #eef4fb !important;
}

#DEX-hero {
    width: 100%;
    position: relative;
    overflow: hidden;
    padding: 34px 38px;
    margin-bottom: 18px;

    border-radius: 24px;
    border: 1px solid rgba(255,255,255,.10);

    background:
        radial-gradient(circle at 92% 6%, rgba(79,127,199,.14), transparent 28%),
        linear-gradient(135deg, #0d1d30 0%, #132840 58%, #17263d 100%);

    box-shadow:
        0 18px 46px rgba(0,0,0,.22);
}

#DEX-hero h1 {
    margin: 0 0 8px 0 !important;
    color: #ffffff !important;
    font-size: 38px !important;
    font-weight: 800 !important;
    letter-spacing: -.8px !important;
}

#DEX-hero h3 {
    color: #c2cede !important;
    font-size: 18px !important;
    font-weight: 600 !important;
}

#DEX-hero p {
    max-width: 1180px;
    color: #a9bbce !important;
    font-size: 15px !important;
    line-height: 1.8 !important;
}

.soft-card {
    background:
        linear-gradient(
            180deg,
            rgba(17,36,58,.98),
            rgba(12,27,45,.98)
        ) !important;

    border:
        1px solid rgba(255,255,255,.10) !important;

    border-radius:
        18px !important;

    padding:
        18px !important;

    box-shadow:
        0 10px 28px rgba(0,0,0,.15) !important;
}

.status-card {
    background:
        linear-gradient(
            90deg,
            rgba(79,127,199,.09),
            rgba(112,128,190,.07)
        ) !important;

    border:
        1px solid rgba(255,255,255,.10) !important;

    border-radius:
        14px !important;

    padding:
        13px 17px !important;
}

.tabs {
    background:
        transparent !important;
}

.tab-nav {
    gap:
        8px !important;

    border-bottom:
        1px solid rgba(255,255,255,.10) !important;

    padding-bottom:
        5px !important;
}

.tab-nav button {
    background:
        transparent !important;

    color:
        #9eafc3 !important;

    border:
        none !important;

    font-weight:
        650 !important;

    padding:
        11px 14px !important;
}

.tab-nav button:hover {
    color:
        #ffffff !important;

    background:
        rgba(255,255,255,.035) !important;
}

.tab-nav button.selected {
    color:
        #ffffff !important;

    background:
        rgba(79,127,199,.10) !important;

    box-shadow:
        inset 0 -2px 0 #4f7fc7 !important;
}

.gradio-container input,
.gradio-container textarea {
    background:
        #10233a !important;

    color:
        #eef4fb !important;

    -webkit-text-fill-color:
        #eef4fb !important;

    border:
        1px solid rgba(255,255,255,.12) !important;

    border-radius:
        12px !important;

    box-shadow:
        none !important;

    font-size:
        15px !important;
}

.gradio-container input:hover,
.gradio-container textarea:hover {
    border-color:
        rgba(79,127,199,.45) !important;
}

.gradio-container input:focus,
.gradio-container textarea:focus {
    border-color:
        #4f7fc7 !important;

    box-shadow:
        0 0 0 3px rgba(79,127,199,.10) !important;

    outline:
        none !important;
}

.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
    color:
        #8397af !important;

    -webkit-text-fill-color:
        #8397af !important;
}

.gradio-container label,
.gradio-container label span {
    color:
        #c7d5e6 !important;

    -webkit-text-fill-color:
        #c7d5e6 !important;

    font-weight:
        600 !important;
}

.gradio-container button {
    border-radius:
        12px !important;

    font-weight:
        700 !important;
}

.gradio-container button.primary {
    background:
        linear-gradient(
            135deg,
            #3f6fb7,
            #4f7fc7
        ) !important;

    color:
        #ffffff !important;

    border:
        1px solid rgba(255,255,255,.08) !important;

    box-shadow:
        0 8px 20px rgba(63,111,183,.18) !important;
}

.gradio-container button.primary:hover {
    background:
        linear-gradient(
            135deg,
            #4a79bf,
            #5f8fd6
        ) !important;
}

.gradio-container button.secondary {
    background:
        #11243a !important;

    color:
        #eef4fb !important;

    border:
        1px solid rgba(255,255,255,.10) !important;
}

.gradio-container .prose {
    color:
        #c5d2e2 !important;

    line-height:
        1.75 !important;
}

.gradio-container .prose h1,
.gradio-container .prose h2,
.gradio-container .prose h3,
.gradio-container .prose strong {
    color:
        #ffffff !important;
}

.gradio-container .prose blockquote {
    background:
        rgba(79,127,199,.055) !important;

    border-left:
        3px solid #4f7fc7 !important;

    color:
        #b9c7d8 !important;

    padding:
        12px 15px !important;

    border-radius:
        0 10px 10px 0 !important;
}

.gradio-container .prose code {
    color:
        #bcd5f5 !important;

    background:
        rgba(79,127,199,.08) !important;

    border:
        1px solid rgba(79,127,199,.18) !important;

    border-radius:
        7px !important;

    padding:
        3px 7px !important;
}

[data-testid="file"] {
    background:
        #0f2136 !important;

    color:
        #eef4fb !important;

    border:
        1.5px dashed rgba(255,255,255,.16) !important;

    border-radius:
        14px !important;
}

.gradio-container table {
    background:
        #0d1d30 !important;

    border:
        1px solid rgba(255,255,255,.10) !important;
}

.gradio-container th {
    background:
        #12243a !important;

    color:
        #dce7f5 !important;
}

.gradio-container td {
    color:
        #bac8da !important;
}

.gradio-container pre,
.gradio-container .json-holder {
    background:
        #091827 !important;

    color:
        #dbe8f6 !important;
}

::-webkit-scrollbar {
    width:
        9px;

    height:
        9px;
}

::-webkit-scrollbar-track {
    background:
        #07111f;
}

::-webkit-scrollbar-thumb {
    background:
        #34465c;

    border:
        2px solid #07111f;

    border-radius:
        999px;
}

::-webkit-scrollbar-thumb:hover {
    background:
        #465a72;
}

@media (max-width: 900px) {
    .gradio-container {
        padding:
            14px !important;
    }

    #DEX-hero {
        padding:
            26px 23px;

        border-radius:
            18px;
    }

    #DEX-hero h1 {
        font-size:
            29px !important;
    }
}
"""


with gr.Blocks(
    title="DEX | Enterprise AI Consultant",
    fill_width=True,
    css=PRO_CSS,
    theme=gr.themes.Base(),
) as demo:

    thread_state = gr.State("")


    gr.Markdown("""
<div id="DEX-hero">

# DEX
### Enterprise AI Consulting & Decision Support

Describe the project. DEX dynamically selects only the internal specialists required for the case, grounds their analysis in the relevant knowledge base, evaluates market feasibility using a fixed enterprise rubric, and produces a consolidated decision report.

</div>
""")


    with gr.Tabs():

        with gr.Tab("Consultation"):

            with gr.Row():

                with gr.Column(
                    scale=5,
                    elem_classes=["soft-card"]
                ):

                    gr.Markdown(
                        "## Enterprise Case"
                    )

                    company = gr.Textbox(
                        label="Company",
                        value="NajdPay"
                    )

                    industry = gr.Textbox(
                        label="Industry",
                        value="FinTech"
                    )

                    project = gr.Textbox(
                        label="Project *",
                        value="AI Customer Support Platform"
                    )

                    problem = gr.Textbox(
                        label="Problem *",
                        lines=4,
                        value=(
                            "The company receives around 5,000 "
                            "customer support requests every day. "
                            "The current support team cannot respond "
                            "quickly enough during peak hours."
                        ),
                    )

                    requirements = gr.Textbox(
                        label="Requirements",
                        lines=7,
                        value=(
                            "- Arabic and English support\n"
                            "- 24/7 customer service\n"
                            "- RAG over internal company policies\n"
                            "- CRM integration\n"
                            "- Human escalation for complex cases\n"
                            "- Protection of sensitive customer data\n"
                            "- Monitoring and logging\n"
                            "- Scale to 100,000 customers"
                        ),
                    )

                    with gr.Row():

                        budget = gr.Number(
                            label="Budget (SAR)",
                            value=500000,
                            minimum=0
                        )

                        expected_users = gr.Number(
                            label="Expected Users",
                            value=100000,
                            minimum=0,
                            precision=0
                        )

                    additional_context = gr.Textbox(
                        label="Additional Context",
                        lines=3,
                        value=(
                            "The company operates in Saudi Arabia "
                            "and handles sensitive financial customer data."
                        ),
                    )

                    analyze_btn = gr.Button(
                        "Start Enterprise Analysis",
                        variant="primary"
                    )


                with gr.Column(scale=7):

                    run_status = gr.Markdown(
                        "**Status:** Ready for consultation.",
                        elem_classes=["status-card"]
                    )

                    dashboard_md = gr.Markdown(
                        "## Project Feasibility\nRun a consultation to generate the market-oriented assessment.",
                        elem_classes=["soft-card"]
                    )

                    routing_md = gr.Markdown(
                        "### Internal Routing\nThe Router has not selected a team yet.",
                        elem_classes=["soft-card"]
                    )

                    report_md = gr.Markdown(
                        "## Final Report\nThe consolidated report will appear here after analysis.",
                        elem_classes=["soft-card"]
                    )


            gr.Markdown("---")


            with gr.Row():

                with gr.Column(
                    scale=5,
                    elem_classes=["soft-card"]
                ):

                    approval_md = gr.Markdown(
                        "### Human Review\nNot started"
                    )

                    revision_feedback = gr.Textbox(
                        label="Revision Feedback",
                        lines=3,
                        placeholder=(
                            "Example: Make the implementation roadmap "
                            "more conservative and clarify the risk controls."
                        ),
                    )

                    with gr.Row():

                        approve_btn = gr.Button(
                            "Approve Report"
                        )

                        revise_btn = gr.Button(
                            "Request Revision"
                        )


        with gr.Tab("Assessment Results"):

            gr.Markdown(
                "## Consolidated Assessment\n"
                "The internal specialist agents remain behind the scenes. "
                "Their findings are consolidated into Technical and Business assessments."
            )

            with gr.Tabs():

                with gr.Tab("Technical Assessment"):

                    technical_md = gr.Markdown(
                        "Technical assessment will appear after the consultation.",
                        elem_classes=["soft-card"]
                    )

                with gr.Tab("Business Assessment"):

                    business_md = gr.Markdown(
                        "Business assessment will appear after the consultation.",
                        elem_classes=["soft-card"]
                    )


        with gr.Tab("Knowledge Base"):

            gr.Markdown(
                "## Specialist Knowledge Base\n"
                "Upload PDF, TXT, or Markdown references. "
                "Each file is indexed only into the relevant specialist knowledge base."
            )

            with gr.Row():

                ai_files = gr.File(
                    label="AI Knowledge",
                    file_count="multiple",
                    file_types=[".pdf", ".txt", ".md"],
                    type="filepath"
                )

                se_files = gr.File(
                    label="Software Engineering",
                    file_count="multiple",
                    file_types=[".pdf", ".txt", ".md"],
                    type="filepath"
                )

            with gr.Row():

                cs_files = gr.File(
                    label="Computer Science",
                    file_count="multiple",
                    file_types=[".pdf", ".txt", ".md"],
                    type="filepath"
                )

                it_files = gr.File(
                    label="IT Infrastructure",
                    file_count="multiple",
                    file_types=[".pdf", ".txt", ".md"],
                    type="filepath"
                )

            with gr.Row():

                is_files = gr.File(
                    label="Information Systems",
                    file_count="multiple",
                    file_types=[".pdf", ".txt", ".md"],
                    type="filepath"
                )

                risk_files = gr.File(
                    label="Risk / Actuarial",
                    file_count="multiple",
                    file_types=[".pdf", ".txt", ".md"],
                    type="filepath"
                )

            index_btn = gr.Button(
                "Index Documents",
                variant="primary"
            )

            kb_status = gr.Markdown(
                "No new documents indexed in this session.",
                elem_classes=["status-card"]
            )

            rag_evidence_md = gr.Markdown(
                "## Retrieval Evidence\nRun a consultation to see the retrieval query, source files, and the actual retrieved text passed to each specialist.",
                elem_classes=["soft-card"]
            )


        with gr.Tab("Trace & Evidence"):

            gr.Markdown(
                "## Workflow Trace & Evidence\n"
                "Review internal routing, execution order, retry attempts, "
                "market scoring criteria, critical risks, and RAG sources."
            )

            trace_json = gr.JSON(
                label="Workflow Trace",
                value={}
            )


        with gr.Tab("Follow-up Consultation"):

            gr.Markdown(
                "## Follow-up Consultation\n"
                "Ask a new question about the same case. DEX will route the question "
                "to only the relevant internal specialist, perform fresh retrieval from "
                "that specialist's knowledge base, and answer the new question rather "
                "than simply repeating the existing report."
            )

            follow_question = gr.Textbox(
                label="Question",
                lines=3,
                placeholder=(
                    "Examples: What infrastructure should we implement first? "
                    "What changes if the budget drops to 250,000 SAR?"
                ),
            )

            follow_btn = gr.Button(
                "Ask DEX",
                variant="primary"
            )

            follow_answer = gr.Markdown(
                "",
                elem_classes=["soft-card"]
            )


    common_outputs = [
        run_status,
        dashboard_md,
        routing_md,
        report_md,
        technical_md,
        business_md,
        rag_evidence_md,
        trace_json,
        approval_md,
        thread_state,
    ]


    analyze_btn.click(
        fn=run_consultation_ui,
        inputs=[
            company,
            industry,
            project,
            problem,
            requirements,
            budget,
            expected_users,
            additional_context,
        ],
        outputs=common_outputs,
    )


    approve_btn.click(
        fn=approve_ui,
        inputs=[thread_state],
        outputs=common_outputs,
    )


    revise_btn.click(
        fn=revise_ui,
        inputs=[
            thread_state,
            revision_feedback,
        ],
        outputs=common_outputs,
    )


    index_btn.click(
        fn=index_all_ui,
        inputs=[
            ai_files,
            se_files,
            cs_files,
            it_files,
            is_files,
            risk_files,
        ],
        outputs=[kb_status],
    )


    follow_btn.click(
        fn=follow_up_ui,
        inputs=[
            thread_state,
            follow_question,
        ],
        outputs=[
            follow_answer,
            trace_json,
        ],
    )


# Generator progress appears live through the queue.
demo.queue(
    default_concurrency_limit=2
)


/tmp/ipykernel_5626/2615898415.py:483: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Gradio Blocks instance: 5 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7e8c01961ee0>
 |-<gradio.components.textbox.Textbox object at 0x7e8c01b27440>
 |-<gradio.components.textbox.Textbox object at 0x7e8c017eda00>
 |-<gradio.components.textbox.Textbox object at 0x7e8bf997b560>
 |-<gradio.components.textbox.Textbox object at 0x7e8c01909c70>
 |-<gradio.components.number.Number object at 0x7e8bf9969430>
 |-<gradio.components.number.Number object at 0x7e8c0047d6d0>
 |-<gradio.components.textbox.Textbox object at 0x7e8bf99ba510>
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x7e8c01b26db0>
 |-<gradio.components.markdown.Markdown object at 0x7e8c00448320>
 |-<gradio.components.markdown.Markdown object at 0x7e8c0047d010>
 |-<gradio.components.markdown.Markdown object at 0x7e8c0044b1a0>
 |-<gradio.components.markdown.Markdown object at 0x7e8bf996e9c0>
 |-<gradio.components.markdown.Markdown obje

In [ ]:
# 14) Launch the colorful dynamic UI
# Open the gradio.live link printed below for the most reliable Colab experience.
demo.launch(share=True, debug=True, show_error=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1176028751b12a1b68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Presentation checklist

1. Index at least one real reference document in two relevant knowledge bases.
2. Show the detailed indexing result: file names and chunk counts.
3. Run the enterprise case and show the Router-selected internal specialists.
4. Show the market-oriented Project Feasibility Score and its seven weighted criteria.
5. Open Assessment Results and show Technical Assessment and Business Assessment.
6. Open Knowledge Base and show the Retrieval Evidence actually used by the consultation.
7. Open Trace & Evidence and show routing, retries, rubric scores, critical risks, and RAG sources.
8. Request one revision, review the updated report, then approve it.
9. Ask a new follow-up question and show that DEX performs new routing and RAG retrieval instead of repeating the report.
